# G01 · Gate 1: C2 image-matched pooling on the development events

Decides whether C2 (projected-footprint pooling) goes to the locked test as RQ2. **Development events only.**

**C2 as run here (fixed before any C2 AUROC is computed):** each tile's footprints are moved by that tile's
measured footprint-to-SAR offset (cross-correlation of the footprints with the SAR image, as in c07; no labels),
but only where the offset is real: a clear peak (peak_z >= 3) that also beats the permutation null. Other tiles
are pooled as drawn. Outline pooling otherwise; DINOv3 ViT-B/16, native resolution.

**Gate 1 (pre-registered, unchanged):** over the 5 development events, (i) mean AUROC gain over outline
pooling >= +0.02, (ii) better than outline on >= 4 of 5 events, (iii) mean gain above the 95th percentile of
the random-shift null: R = 10 draws, each moving every tile by the SAME distance as C2 in a random direction
(`stats/gates.py`). Also reported, not part of the gate: C2 against dilated 3 m and against the 10 m ring.

* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN`.
  Needs p01 (building table), p02 (DINOv3 outline + dilated features) and p03 (DINOv3 ring features).
* **Time:** about 60–80 minutes (offsets ~10 min CPU, two encoding passes ~25 min, bench run ~30 min); ~3 units.
* **Outputs:** `offsets.csv`, `shift_summary.csv`, the bench run and `gate1.json` in this run's folder.

In [ ]:
NOTEBOOK_NAME = "g01_gate1_c2_matched_pooling"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU, Hugging Face token, inputs from p02 and p03
import glob, os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
P02_FEATS = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
RING = sorted(glob.glob(f"{PROJECT}/02_models/features/*_controls_imagenet/dinov3_ring.parquet"))
assert os.path.exists(P02_FEATS), f"missing {P02_FEATS}: run p02 first"
assert RING, "missing the p03 DINOv3 ring features: run p03 first"
RING = RING[-1]
print("GPU:", torch.cuda.get_device_name(0), "| ring features:", RING)

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import datetime, gc
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_c2_matched"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# 1. Offsets per tile (as c07; footprints + SAR only, no labels) -> the C2 shifts, fixed before any AUROC
from damagesar_bench.c2_offsets import event_summary, matched_shifts, measure_offsets
from damagesar_bench.encoders.extract import read_bright_tile
dev_events = [e for e, c in events.events.items() if c.role in ("development", "secondary")]
off = measure_offsets(use, lambda t: read_bright_tile(ROOT, t), dev_events, factor=4, max_shift_m=40, n_null=3,
                      seed=0, log=print)
off.to_csv(f"{RUN}/offsets.csv", index=False)
SHIFTS = matched_shifts(off, peak_z_min=3.0)
moved = pd.Series({t: (dy, dx) != (0.0, 0.0) for t, (dy, dx) in SHIFTS.items()})
summ = off.assign(moved=off["tile_id"].map(moved)).groupby("event").agg(
    tiles=("tile_id", "size"), tiles_moved=("moved", "sum"),
    median_shift_m=("magnitude_m", lambda s: float(s[off.loc[s.index, "tile_id"].map(moved)].median())))
summ.to_csv(f"{RUN}/shift_summary.csv")
display(event_summary(off).round(3)); display(summ.round(2))

In [ ]:
# 2. DINOv3 features with C2 (matched) and the 10 random-direction draws, native resolution, two passes
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
GROUPS = {"a": ["matched"] + [f"rshift:{k}" for k in range(0, 5)], "b": [f"rshift:{k}" for k in range(5, 10)]}
FEATS, timing = [], []
enc = get_encoder("dinov3_vitb16")
print("weights:", enc.weights_revision)
for g, methods in GROUPS.items():
    f, runs = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, methods, [None],
                                     f"/content/features/c2_{g}", events, log=print, shifts=SHIFTS)
    FEATS.append(str(write_features(f, f"{FEAT_DIR}/dinov3_c2_{g}.parquet")))
    timing.append(runs_table(runs)); del f; gc.collect()
del enc; torch.cuda.empty_cache()
pd.concat(timing).to_csv(f"{RUN}/timing.csv", index=False)

In [ ]:
# 3. bench run: C2 and every random draw against outline; C2 against dilated and ring (reported, not gated)
del use; gc.collect()
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
D = "dinov3_vitb16@native"
comps = [ComparisonSpec((D, "matched"), (D, "outline"), "C2 matched vs outline"),
         ComparisonSpec((D, "matched"), (D, "dilated3m"), "C2 matched vs dilated 3 m"),
         ComparisonSpec((D, "matched"), (D, "ring10m"), "C2 matched vs ring 10 m")]
comps += [ComparisonSpec((D, f"rshift{k}"), (D, "outline"), f"random shift {k} vs outline") for k in range(10)]
out = run_benchmark([P02_FEATS, RING, *FEATS], comps, events, f"{RUN}/bench_run", n_boot=2000, only_compared=True)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))

In [ ]:
# 4. Gate 1 verdict (pre-registered rule, stats/gates.py)
import json
import numpy as np
from damagesar_bench.stats.gates import gate1
c = pd.read_csv(f"{out}/comparisons.csv").set_index("label")
gains = json.loads(c.loc["C2 matched vs outline", "gains"])
null = [float(np.mean(list(json.loads(c.loc[f"random shift {k} vs outline", "gains"]).values()))) for k in range(10)]
g = gate1(gains, null)
verdict = {"gains_by_event": gains, "mean_gain": g.mean_gain, "n_better": g.n_better, "n_events": g.n_events,
           "null_mean_gains": null, "null_threshold_p95": g.null_threshold, "mean_ok": g.mean_ok,
           "count_ok": g.count_ok, "null_ok": g.null_ok, "passed": g.passed}
json.dump(verdict, open(f"{RUN}/gate1.json", "w"), indent=2)
print(json.dumps(verdict, indent=2))
print("GATE 1:", "PASSED" if g.passed else "NOT PASSED")
m = pd.read_csv(f"{out}/event_metrics.csv")
pv = m.pivot_table(index="pooling", columns="event", values="auroc"); pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string())

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))